In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 通过load_dotenv()将.env中的变量加载为环境变量
# override=True表示：无论你当前的操作系统、终端或者虚拟环境中是否已经存在同名的环境变量，
# 都会强行用 .env 文件里写的值去覆盖它
load_dotenv(override=True)


# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


# 创建LLM
model = init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model="deepseek:deepseek-flash" # 冒号前面直定模型厂商，冒号后面指定模型
)

resp = model.invoke('你好')
resp.pretty_print()

================================== Ai Message ==================================

你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是聊天、解答问题、写作、翻译，还是其他任何事情，都可以告诉我。


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True, # 不管是approve, reject, edit都拒绝
                "get_news": True,
                "read_email_tool": False, # 工具随便调，不会阻止
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"], # 会进行中断
                    "description": "发送邮件中断了..." # 如果这个中断被触发，描述信息就是这个
                },
            },
            description_prefix="中断啦！！" # 如果上面工具被中断，没有默认描述则用这个全局的中断信息
        ),
    ]
)

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({
    "messages": [HumanMessage(content="请帮我查询今天北京的天气"
                                      "查询今日新闻"
                                      "查看ID为 'sk2131421' 的邮件内容，"
                                      "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                                      "同时做这四件事")]
},
    config=config
)


rprint(response)

{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='0a3fbbea-4eba-4553-aa08-e053da03625b'
        ),
        AIMessage(
            content='我来同时处理这四件事：',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': "The user wants me to do four things in parallel:\n1. Query today's Beijing 
weather\n2. Query today's news\n3. Read email with ID 'sk2131421'\n4. Send email to 15641685664@qq.com with subject
'哈哈哈' and body '你好啊'\n\nThese are independent, so I can do them all in one block."
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 238,
                    'prompt_tokens': 532,
                    'total_tokens': 770,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 76,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 532
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '890deffa-4ae9-4906-84f5-49e58e3d1dd3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a14d8d-1dd4-7b82-88fd-3c05651f4abf-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_Iz9sJQkJVXIh2q6eYsdO2335',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_unN2KGMxN9trsCTozKQf7568', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_gd6tVXoSXC3hBHJgZvjd8019',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_D6UPLgVCBRHQfRMKDJN46380',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 532,
                'output_tokens': 238,
                'total_tokens': 770,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 76}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京'},
                        'description': "中断啦！！\n\nTool: get_weather\nArgs: {'city': '北京'}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦！！\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断了...'
                    }
             

配置了中断之后，下面我们要根据中断的工具配置选择了

In [3]:
from langgraph.types import Command

# 前面配置了
weather_decision = {
    "type" : "edit",
    "edited_action" : {
        "name" : "get_weather",
        "args" : {"city" : "上海市","is_forcast" : True},
    }
}


news_decision = {
    "type" : "approve"
}


send_email_decision = {
    "type" : "reject"
}



decisions = {
    "decisions" : []
}

interrupts = response.get("__interrupt__",[])
action_requests = interrupts[0].value["action_requests"]


for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts :
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config = config
    )

    for msg in resumed_response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我来同时处理这四件事：
Tool Calls:
  get_weather (call_00_Iz9sJQkJVXIh2q6eYsdO2335)
 Call ID: call_00_Iz9sJQkJVXIh2q6eYsdO2335
  Args:
    city: 北京
  get_news (call_01_unN2KGMxN9trsCTozKQf7568)
 Call ID: call_01_unN2KGMxN9trsCTozKQf7568
  Args:
  read_email_tool (call_02_gd6tVXoSXC3hBHJgZvjd8019)
 Call ID: call_02_gd6tVXoSXC3hBHJgZvjd8019
  Args:
    email_id: sk2131421
  send_email_tool (call_03_D6UPLgVCBRHQfRMKDJN46380)
 Call ID: call_03_D6UPLgVCBRHQfRMKDJN46380
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: send_email_tool

User rejected the tool call for `send_email_tool` with id call_03_D6UPLgVCBRHQfRMKDJN46380. The tool was not 